# Inference, one stage at a time

`03_benchmark.ipynb` runs a whole tile in one call and gives numbers. This notebook runs **one tile** through the same code and stops after every stage to show a picture:

1. Remove the ground
2. Find clicks
3. Point-SAM: one mask per click
4. Merge the masks
5. Score against the true trees

**With a GPU** (ICE) it runs everything live. **Without one** (a Mac) it skips stage 3 and loads the masks that `03_benchmark.ipynb` saved when run with `save_masks: true`, so stages 1, 2, 4 and 5 still run for real. Copy `results/diagnose_val/*_masks.npz` and `*_pred.npy` from ICE first.

Use a **val** tile. Looking closely at test tiles and then changing settings would spoil the test strip.

In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

from treeseg.data.ground import classify, plot_slice
from treeseg.engine.benchmark import prepare_tile
from treeseg.engine.clicks import make_clicks
from treeseg.engine.scoring import funnel, match_instances, score_instances, score_raw_masks
from treeseg.engine.segment import click_candidates, fuse_masks, load_masks, predict_click_masks, save_masks
from treeseg.utils.view3d import (click_status, plot_click_map, plot_crop_sampling, plot_mask_candidates, plot_merge, plot_outcome,
                                  plot_outcome_3d, plot_tree_gallery)

cfg = yaml.safe_load(open(os.environ.get("TREESEG_CONFIG", "configs/experiment/diagnose_val.yaml")))
CHECKPOINT = list(cfg["checkpoints"])[-1]
TILE = cfg["tiles"][0]
CLICKS = "treex"  # perfect, treex or stems

LIVE = torch.cuda.is_available()
masks_path = os.path.join(cfg["out_dir"], f"{CHECKPOINT}_{TILE}_{CLICKS}_masks.npz")
figures = os.path.join(cfg["out_dir"], "inspect")
os.makedirs(figures, exist_ok=True)
name = f"{CHECKPOINT}_{TILE}_{CLICKS}"
print("running live on", torch.cuda.get_device_name(0) if LIVE else f"no GPU: stage 3 comes from {masks_path}")

## Stage 1: remove the ground

Clicks start low on trunks, so leftover ground attracts them. A side view of one thin slice: blue was removed as ground, grey is tree points that were kept, and orange is kept points that are not a tree (understory, or ground the filter missed). A clean result has blue along the bottom and no orange band there. Tree sizes are counted **before** removal, so a tree that loses points here still has to be found whole.

In [ ]:
tile = prepare_tile(cfg["tile_dir"], TILE, cfg["csf"], cfg["min_tree_points"])
xyz, ids = tile["xyz"], tile["ids"]
print(f"{len(tile['full_xyz']):,} points in the tile, {len(xyz):,} after ground removal, {len(tile['sizes'])} trees to find")
full_ids = np.full(len(tile["full_xyz"]), -1)
full_ids[tile["keep"]] = ids
plot_slice(tile["full_xyz"], classify(~tile["keep"], full_ids), f"{TILE}: ground removal")
plt.show()

## Stage 2: find clicks

From above. Each circle is a true tree's base: green has a click on it, red has none. Diamonds are clicks on a tree and crosses are clicks that missed. A tree with no click can only be found by luck, through a neighbour's mask.

In [ ]:
if LIVE or not os.path.exists(masks_path):
    clicks = make_clicks(CLICKS, xyz, ids, tile["full_xyz"])
else:
    clicks = load_masks(masks_path)["clicks"]  # the exact clicks the saved masks were made from
plot_click_map(xyz, ids, clicks, f"{TILE}: {len(clicks)} {CLICKS} clicks", cfg["min_tree_points"])
plt.savefig(os.path.join(figures, f"{name}_clicks.png"), bbox_inches="tight")
plt.show()

## Stage 3a: one click, as the model sees it

The model never sees the tile. For each click it gets a 12 m square column around it, thinned to 10,000 points, and returns 3 candidate masks with a confidence score each. Inference keeps the one it scores highest, because there is no true answer to check against.

Change `CLICK_NUMBER` to look at other clicks. **Needs a GPU.**

In [ ]:
CLICK_NUMBER = 0

if LIVE:
    from treeseg.models.point_sam import build_point_sam

    model = build_point_sam(cfg["checkpoints"][CHECKPOINT], cfg["group_number"], cfg["group_size"]).to("cuda").eval()
    one = click_candidates(model, xyz, clicks[CLICK_NUMBER], cfg["crop_size"], cfg["crop_points"])
    plot_crop_sampling(xyz, one, f"Click {CLICK_NUMBER}: the crop")
    plt.show()
    tree = click_status(xyz, ids, [clicks[CLICK_NUMBER]])[0][0]
    truth = ids[one["sample_index"]] == tree
    #IoU inside the crop sample only, so a tree cut by the crop edge can still score well here
    ious = [(mask & truth).sum() / max((mask | truth).sum(), 1) for mask in one["masks"]]
    plot_mask_candidates({"coords": xyz[one["sample_index"]], "truth": truth, "masks": one["masks"], "scores": one["scores"], "ious": ious,
                          "picked": one["picked"], "click": one["click"]}, f"Click {CLICK_NUMBER} landed on tree {tree}" if tree > 0 else f"Click {CLICK_NUMBER} missed every tree")
    plt.savefig(os.path.join(figures, f"{name}_click{CLICK_NUMBER}.png"), bbox_inches="tight")
    plt.show()
else:
    print("no GPU: skipped. Run this cell on ICE to see the 3 candidate masks.")

## Stage 3b: every click

The same thing for all clicks: one raw mask each. `raw_recall` is the share of trees where some single mask already reaches IoU 0.5, before any merging.

In [ ]:
if LIVE:
    raw_masks = predict_click_masks(model, xyz, clicks, crop_size=cfg["crop_size"], crop_stride=cfg["crop_stride"], crop_points=cfg["crop_points"])
    save_masks(masks_path, clicks, *raw_masks)
    masks, logits, scores, seeds = raw_masks
else:
    saved = load_masks(masks_path)
    masks, logits, scores, seeds = saved["masks"], saved["logits"], saved["scores"], saved["seeds"]
raw = score_raw_masks(ids, masks, tile["sizes"], cfg["iou_threshold"])
print(f"{len(masks)} raw masks. {raw['raw_trees_found']} of {len(tile['sizes'])} trees have a good one (raw_recall {raw['raw_recall']:.3f})")

## Stage 4: merge the masks

Hundreds of overlapping masks become one label per point. `fuse_masks` does three things:

- **Joins** two masks if each contains the other's click, or one sits at least `CONTAINMENT` inside the other. Joins chain, so A joined to B and B joined to C makes one tree.
- **Settles overlaps:** a point claimed by several trees goes to the most confident one.
- **Drops** anything left with fewer than `MIN_POINTS` points.

Change the two settings and rerun from here to see their effect. This stage needs no GPU.

In [ ]:
CONTAINMENT, MIN_POINTS = 0.8, 50

pred, groups = fuse_masks(len(xyz), masks, logits, scores, seeds, containment=CONTAINMENT, min_points=MIN_POINTS, return_groups=True)
plot_merge(xyz, masks, groups, pred, f"{name}: merging with containment {CONTAINMENT}")
plt.savefig(os.path.join(figures, f"{name}_merge.png"), bbox_inches="tight")
plt.show()

## Stage 5: score against the true trees

Each true tree is paired with at most one prediction, and a pair counts when its IoU is 0.5 or more. A matched tree has the **same colour** on both sides. Dark trees on the left were not matched. Red on the right is a prediction that matches no tree, and tan is a prediction on understory, which is not counted against the model.

In [ ]:
match = match_instances(ids, pred, tile["sizes"], cfg["iou_threshold"], ignore_non_tree_preds=True)
score = score_instances(ids, pred, tile["sizes"], cfg["iou_threshold"], ignore_non_tree_preds=True)
plot_outcome(xyz, ids, pred, match, clicks, f"{name}: F1 {score['f1']:.3f}, precision {score['precision']:.3f}, recall {score['recall']:.3f}")
plt.savefig(os.path.join(figures, f"{name}_outcome.png"), bbox_inches="tight")
plt.show()

### Four trees up close

Green is the part of the tree that was found, dark is the part that was missed, red is where the prediction spilled onto something else.

In [ ]:
plot_tree_gallery(xyz, ids, pred, match, name)
plt.savefig(os.path.join(figures, f"{name}_gallery.png"), bbox_inches="tight")
plt.show()

### Where were the trees lost?

How many trees are left after each stage. A big drop between two rows names the stage to fix.

In [ ]:
lost = pd.DataFrame(funnel(xyz, ids, tile["sizes"], clicks, raw, match))
lost.to_csv(os.path.join(figures, f"{name}_funnel.csv"), index=False)
lost

### Rotate it

Left is the truth, right is the prediction, with the same colours as above. A 80,000-point sample.

In [ ]:
plot_outcome_3d(xyz, ids, pred, match, name)

## The baseline on the same tile

TreeX segmenting the tile by itself, with no model, scored the same way. Shown if `03_benchmark.ipynb` ran with `baselines: [treex_full]`.

In [ ]:
baseline_path = os.path.join(cfg["out_dir"], f"treex_full_{TILE}_none_pred.npy")
if os.path.exists(baseline_path):
    baseline = np.load(baseline_path)
    baseline_match = match_instances(ids, baseline, tile["sizes"], cfg["iou_threshold"], ignore_non_tree_preds=True)
    baseline_score = score_instances(ids, baseline, tile["sizes"], cfg["iou_threshold"], ignore_non_tree_preds=True)
    plot_outcome(xyz, ids, baseline, baseline_match, title=f"TreeX alone, {TILE}: F1 {baseline_score['f1']:.3f}, precision {baseline_score['precision']:.3f}, recall {baseline_score['recall']:.3f}")
    plt.savefig(os.path.join(figures, f"treex_full_{TILE}_outcome.png"), bbox_inches="tight")
    plt.show()
else:
    print("no baseline prediction saved for this tile")